<a href="https://colab.research.google.com/github/abdul2924/Machine_learning_Assignment/blob/main/model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Load Data for Model Training and Comparison
BASE_PATH = "/content/drive/MyDrive/yellow_tripdata_2019"

from pyspark.sql import SparkSession
from pyspark.ml.regression import LinearRegression, GBTRegressor, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
import pyspark.sql.functions as F

spark = SparkSession.builder.getOrCreate()
print(f"Spark {spark.version} ready")

train_df = spark.read.parquet(f"{BASE_PATH}/data/final/train_2019")
val_df = spark.read.parquet(f"{BASE_PATH}/data/final/val_2019")
test_df = spark.read.parquet(f"{BASE_PATH}/data/final/test_2019")

print(f"Train: {train_df.count():,} rows | Schema: {train_df.columns}")
print(f"Val:   {val_df.count():,} rows")
print(f"Test:  {test_df.count():,} rows")
train_df.show(3, truncate=False)

Spark 4.0.2 ready
Train: 68,360,825 rows | Schema: ['features', 'fare_amount']
Val:   6,603,163 rows
Test:  6,617,319 rows
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|features                                                                                                                                                                                                                                                |fare_amount|
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|[1.1699727455587712,1.8139108170292353,0.0,0.1388096899939466,-2.3068364200405878,-0.76

In [3]:
# Define Regression Model Configurations
from pyspark.ml.regression import (
    LinearRegression,
    GBTRegressor,
    RandomForestRegressor
)


models_config = {

    "1. Linear Regression": LinearRegression(
        featuresCol="features",
        labelCol="fare_amount",
        maxIter=50,
        regParam=0.1,
        elasticNetParam=0.5
    ),

    "2. Gradient Boosted Trees": GBTRegressor(
        featuresCol="features",
        labelCol="fare_amount",
        maxIter=20,
        maxDepth=6,
        stepSize=0.05
    ),

    "3. Random Forest": RandomForestRegressor(
        featuresCol="features",
        labelCol="fare_amount",
        numTrees=20,
        maxDepth=8,
        featureSubsetStrategy="sqrt"
    ),

    "4. Ridge Regression": LinearRegression(
        featuresCol="features",
        labelCol="fare_amount",
        maxIter=50,
        regParam=1.0,
        elasticNetParam=0.0
    )
}

print("4 MLlib Regression Models:")
for name in models_config.keys():
    print(f"  • {name}")

4 MLlib Regression Models:
  • 1. Linear Regression
  • 2. Gradient Boosted Trees
  • 3. Random Forest
  • 4. Ridge Regression


In [4]:
# Train and Evaluate Multiple Models on Sampled Data
from pyspark.ml.evaluation import RegressionEvaluator


train_small = (
    train_df
        .select("features", "fare_amount")
        .sample(withReplacement=False, fraction=0.01, seed=42)
        .cache()
)

val_small = (
    val_df
        .select("features", "fare_amount")
        .sample(withReplacement=False, fraction=0.05, seed=42)
        .cache()
)

print(f"Train small: {train_small.count():,} rows")
print(f"Val small:   {val_small.count():,} rows")


evaluator_rmse = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)

evaluator_r2 = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="r2"
)

trained_models = {}
val_scores = []


for name, model in models_config.items():

    print(f"\nTraining {name}...")

    model_trained = model.fit(train_small)
    trained_models[name] = model_trained

    val_pred = model_trained.transform(val_small)

    rmse = evaluator_rmse.evaluate(val_pred)
    r2 = evaluator_r2.evaluate(val_pred)

    val_scores.append((name, __builtins__.round(rmse, 3), __builtins__.round(r2, 4)))

    print(f"RMSE: ${rmse:.2f} | R²: {r2:.4f}")


results_df = spark.createDataFrame(
    val_scores,
    ["Model", "RMSE", "R2"]
)

print("\nValidation Results (Sorted by RMSE):")
results_df.orderBy("RMSE").show(truncate=False)


train_small.unpersist()
val_small.unpersist()

Train small: 684,994 rows
Val small:   330,009 rows

Training 1. Linear Regression...
RMSE: $4.19 | R²: 0.8657

Training 2. Gradient Boosted Trees...
RMSE: $4.03 | R²: 0.8756

Training 3. Random Forest...
RMSE: $4.09 | R²: 0.8723

Training 4. Ridge Regression...
RMSE: $4.27 | R²: 0.8605

Validation Results (Sorted by RMSE):
+-------------------------+-----+------+
|Model                    |RMSE |R2    |
+-------------------------+-----+------+
|2. Gradient Boosted Trees|4.035|0.8756|
|3. Random Forest         |4.089|0.8723|
|1. Linear Regression     |4.193|0.8657|
|4. Ridge Regression      |4.274|0.8605|
+-------------------------+-----+------+



DataFrame[features: vector, fare_amount: double]

In [5]:
# Final Evaluation of Models on Test Data
from pyspark.ml.evaluation import RegressionEvaluator


test_data = test_df.select("features", "fare_amount")


evaluator_rmse = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)

evaluator_mae = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="mae"
)

evaluator_r2 = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="r2"
)

print(f"{'Model':<25} {'Test RMSE ($)':<15} {'Test MAE ($)':<15} {'Test R²':<10}")

test_results = {}

for name, model in trained_models.items():

    print(f"\nTest predictions: {name}")

    test_pred = model.transform(test_data)

    rmse = evaluator_rmse.evaluate(test_pred)
    mae = evaluator_mae.evaluate(test_pred)
    r2 = evaluator_r2.evaluate(test_pred)

    test_results[name] = {
        "Test_RMSE": __builtins__.round(rmse, 3),
        "Test_MAE": __builtins__.round(mae, 3),
        "Test_R2": __builtins__.round(r2, 4)
    }

    print(f"  RMSE: ${rmse:.2f} | MAE: ${mae:.2f} | R²: {r2:.4f}")


print("\nFINAL RANKING:")

sorted_results = sorted(
    test_results.items(),
    key=lambda x: x[1]["Test_RMSE"]
)

for i, (name, scores) in enumerate(sorted_results, start=1):
    print(
        f"{i}. {name:<23} "
        f"RMSE=${scores['Test_RMSE']:6.2f}  "
        f"MAE=${scores['Test_MAE']:6.2f}  "
        f"R²={scores['Test_R2']:7.4f}"
    )

best_model_name = sorted_results[0][0]
print(f"\nBEST MODEL: {best_model_name}")

Model                     Test RMSE ($)   Test MAE ($)    Test R²   

Test predictions: 1. Linear Regression
  RMSE: $4.31 | MAE: $2.15 | R²: 0.8635

Test predictions: 2. Gradient Boosted Trees
  RMSE: $4.07 | MAE: $0.99 | R²: 0.8781

Test predictions: 3. Random Forest
  RMSE: $4.13 | MAE: $1.19 | R²: 0.8746

Test predictions: 4. Ridge Regression
  RMSE: $4.39 | MAE: $2.23 | R²: 0.8582

FINAL RANKING:
1. 2. Gradient Boosted Trees RMSE=$  4.07  MAE=$  0.99  R²= 0.8781
2. 3. Random Forest        RMSE=$  4.13  MAE=$  1.19  R²= 0.8746
3. 1. Linear Regression    RMSE=$  4.31  MAE=$  2.15  R²= 0.8635
4. 4. Ridge Regression     RMSE=$  4.39  MAE=$  2.23  R²= 0.8582

BEST MODEL: 2. Gradient Boosted Trees


In [6]:
# Hyperparameter Tuning with Cross-Validation
from pyspark.ml.regression import GBTRegressor, RandomForestRegressor, LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

print(f"\nTUNING {best_model_name}...")


if "Gradient Boosted Trees" in best_model_name:
    model_tune = GBTRegressor(
        featuresCol="features",
        labelCol="fare_amount"
    )

    paramGrid = (
        ParamGridBuilder()
            .addGrid(model_tune.maxDepth, [3, 5])
            .addGrid(model_tune.maxIter, [10, 20])
            .build()
    )

elif "Random Forest" in best_model_name:
    model_tune = RandomForestRegressor(
        featuresCol="features",
        labelCol="fare_amount"
    )

    paramGrid = (
        ParamGridBuilder()
            .addGrid(model_tune.maxDepth, [4, 6])
            .addGrid(model_tune.numTrees, [10, 20])
            .build()
    )

else:
    model_tune = LinearRegression(
        featuresCol="features",
        labelCol="fare_amount"
    )

    paramGrid = (
        ParamGridBuilder()
            .addGrid(model_tune.regParam, [0.01, 0.1])
            .addGrid(model_tune.maxIter, [50, 100])
            .build()
    )


train_cv_sample = (
    train_df
        .select("features", "fare_amount")
        .sample(withReplacement=False, fraction=0.01, seed=42)
        .cache()
)

train_cv_sample.count()


evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)


cv = CrossValidator(
    estimator=model_tune,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=2
)

print("Running 2-Fold Cross-Validation on sampled data...")

cv_model = cv.fit(train_cv_sample)


test_data = test_df.select("features", "fare_amount")
cv_test_pred = cv_model.bestModel.transform(test_data)

cv_rmse = evaluator.evaluate(cv_test_pred)

print("\nBest Parameters:")
print(cv_model.bestModel.extractParamMap())

print(f"\nTuned Test RMSE: ${cv_rmse:.2f}")


train_cv_sample.unpersist()


TUNING 2. Gradient Boosted Trees...
Running 2-Fold Cross-Validation on sampled data...

Best Parameters:
{Param(parent='GBTRegressor_8bbf332fe6b5', name='cacheNodeIds', doc='If false, the algorithm will pass trees to executors to match instances with nodes. If true, the algorithm will cache node IDs for each instance. Caching can speed up training of deeper trees. Users can set how often should the cache be checkpointed or disable it by setting checkpointInterval.'): False, Param(parent='GBTRegressor_8bbf332fe6b5', name='checkpointInterval', doc='set checkpoint interval (>= 1) or disable checkpoint (-1). E.g. 10 means that the cache will get checkpointed every 10 iterations. Note: this setting will be ignored if the checkpoint directory is not set in the SparkContext.'): 10, Param(parent='GBTRegressor_8bbf332fe6b5', name='featureSubsetStrategy', doc="The number of features to consider for splits at each tree node. Supported options: 'auto' (choose automatically for task: If numTrees =

DataFrame[features: vector, fare_amount: double]

In [7]:
# Analyze Feature Importance from Tree-based Model
best_tree_model = None
best_tree_model_name = None


for name, model in trained_models.items():
    if "Gradient Boosted Trees" in name or "Random Forest" in name:
        best_tree_model = model
        best_tree_model_name = name
        break


if best_tree_model and hasattr(best_tree_model, "featureImportances"):

    importances = best_tree_model.featureImportances.toArray()


    features = [
        "passenger", "trip_dist", "haversine", "duration",
        "hour", "peak_hr", "ratecode", "short_trip",
        "congestion", "airport"
    ]

    print(f"\nFEATURE IMPORTANCE ({best_tree_model_name}):")
    print(f"{'Feature':<15} {'Importance'}")

    for feat, imp in zip(features, importances):
        print(f"{feat:<15} {imp:.4f}")

else:
    print("\nSelected model is Linear Regression - no feature importance available.")


FEATURE IMPORTANCE (2. Gradient Boosted Trees):
Feature         Importance
passenger       0.0030
trip_dist       0.8089
haversine       0.0000
duration        0.1636
hour            0.0170
peak_hr         0.0007
ratecode        0.0011
short_trip      0.0004
congestion      0.0008
airport         0.0003


In [8]:
# Perform Residual Analysis
from pyspark.sql import functions as F


test_data = test_df.select("features", "fare_amount")


best_model = trained_models[best_model_name]


test_final = best_model.transform(test_data)


test_final = test_final.withColumn(
    "residual",
    F.col("fare_amount") - F.col("prediction")
)


test_final.select(
    "prediction",
    "fare_amount",
    "residual"
).show(10, truncate=False)


residual_summary = test_final.select(
    F.round(F.avg("residual"), 2).alias("mean_residual"),
    F.round(F.stddev("residual"), 2).alias("residual_std")
)

print("\nResidual Summary Statistics:")
residual_summary.show()

+------------------+-----------+---------------------+
|prediction        |fare_amount|residual             |
+------------------+-----------+---------------------+
|13.583963773054228|13.0       |-0.5839637730542275  |
|4.863391916380289 |5.0        |0.136608083619711    |
|14.185790339531497|13.5       |-0.685790339531497   |
|4.534605763932155 |4.5        |-0.034605763932154865|
|16.739714396266905|16.5       |-0.23971439626690483 |
|6.258349012578924 |6.0        |-0.2583490125789236  |
|5.724122819935032 |5.5        |-0.22412281993503225 |
|9.737922024997735 |9.5        |-0.23792202499773474 |
|7.370855257206177 |7.5        |0.12914474279382304  |
|5.315249114881951 |5.5        |0.18475088511804927  |
+------------------+-----------+---------------------+
only showing top 10 rows

Residual Summary Statistics:
+-------------+------------+
|mean_residual|residual_std|
+-------------+------------+
|         0.03|        4.07|
+-------------+------------+



In [9]:
# Save Best Models and Export Results for Visualization
from pyspark.sql import functions as F


safe_model_name = best_model_name.replace(" ", "_").lower()


best_model.write().overwrite().save(
    f"{BASE_PATH}/tests/{safe_model_name}"
)


cv_model.bestModel.write().overwrite().save(
    f"{BASE_PATH}/tests/tuned_best_model"
)


test_data = test_df.select("features", "fare_amount")


test_final = best_model.transform(test_data)


test_final_with_residual = test_final.withColumn(
    "residual",
    F.col("fare_amount") - F.col("prediction")
)


(
    test_final_with_residual
        .select("prediction", "fare_amount", "residual")
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(f"{BASE_PATH}/tableau/final_predictions")
)


print(f"Best model saved at: {BASE_PATH}/tests/{safe_model_name}")
print(f"Tuned model saved at: {BASE_PATH}/tests/tuned_best_model")
print(f"Tableau export saved at: {BASE_PATH}/tableau/final_predictions")


print("\nBEST RESULTS:")
if best_model_name in test_results:
    scores = test_results[best_model_name]
    print(
        f"{best_model_name}: "
        f"RMSE=${scores['Test_RMSE']:.2f}, "
        f"R²={scores['Test_R2']:.4f}"
    )

Best model saved at: /content/drive/MyDrive/yellow_tripdata_2019/tests/2._gradient_boosted_trees
Tuned model saved at: /content/drive/MyDrive/yellow_tripdata_2019/tests/tuned_best_model
Tableau export saved at: /content/drive/MyDrive/yellow_tripdata_2019/tableau/final_predictions

BEST RESULTS:
2. Gradient Boosted Trees: RMSE=$4.07, R²=0.8781


In [10]:
# Export Model Metrics to CSV for Tableau
from pyspark.sql import SparkSession
from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd
from pyspark.ml.regression import GBTRegressionModel
spark = SparkSession.builder.getOrCreate()


BASE_PATH = "/content/drive/MyDrive/yellow_tripdata_2019"


best_model_name = "2. Gradient Boosted Trees"


if 'trained_models' in globals() and best_model_name in trained_models:
    best_model = trained_models[best_model_name]
else:
    print(f"Warning: 'trained_models' or '{best_model_name}' not found in current scope. Attempting to load best model from disk.")

    safe_model_name = best_model_name.replace(" ", "_").lower()
    model_path = f"{BASE_PATH}/tests/{safe_model_name}"
    try:

        best_model = GBTRegressionModel.load(model_path)
        print(f"Successfully loaded '{best_model_name}' from {model_path}")
    except Exception as e:
        raise RuntimeError(f"Could not load best_model from disk at {model_path}. Ensure previous cells were executed to train and save the model, or check the path. Error: {e}")


test_df = spark.read.parquet(f"{BASE_PATH}/data/final/test_2019")

evaluator_rmse = RegressionEvaluator(labelCol="fare_amount", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol="fare_amount", metricName="mae")
evaluator_r2 = RegressionEvaluator(labelCol="fare_amount", metricName="r2")

predictions = best_model.transform(test_df)

rmse = evaluator_rmse.evaluate(predictions)
mae = evaluator_mae.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

metrics_df = pd.DataFrame({
    "Model": ["Gradient Boosted Trees"],
    "RMSE": [rmse],
    "MAE": [mae],
    "R2": [r2]
})

metrics_df.to_csv(f"{BASE_PATH}/tableau/model_comparison.csv", index=False)

In [11]:
# Export Feature Importance to CSV for Tableau
import pandas as pd


BASE_PATH = "/content/drive/MyDrive/yellow_tripdata_2019"


importances = best_model.featureImportances.toArray()


numeric_features_names = [
    "passenger_count",
    "trip_distance",
    "haversine_km",
    "trip_duration_min",
    "pickup_hour",
    "is_peak_hour",
    "is_weekend"
]


one_hot_encoded_dow_names = [f"dow_encoded_{i}" for i in range(6)]


feature_names = numeric_features_names + one_hot_encoded_dow_names


if len(feature_names) != len(importances):
    print(f"Warning: Mismatch between reconstructed feature names count ({len(feature_names)}) and actual importances count ({len(importances)}). Using generic names as fallback.")
    feature_names = [f"feature_{i}" for i in range(len(importances))]


fi_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

fi_df = fi_df.sort_values("Importance", ascending=False)


fi_df.to_csv(f"{BASE_PATH}/tableau/feature_importance.csv", index=False)

In [12]:
# Export Data Quality Summary to CSV for Tableau
from pyspark.sql.functions import col, count, when
import pandas as pd

total_rows = test_df.count()

null_counts = [
    (c, test_df.filter(col(c).isNull()).count())
    for c in test_df.columns
]

dq_df = pd.DataFrame(null_counts, columns=["Column", "Null_Count"])
dq_df["Null_Percentage"] = dq_df["Null_Count"] / total_rows * 100


dq_df.to_csv(f"{BASE_PATH}/tableau/data_quality_summary.csv", index=False)